In [1]:
import pandas as pd
from pathlib import Path

# caminho dos dados da CVM
BASE_CVM = Path("../data/raw/cvm/dfp")

# arquivos que vamos utilizar
dre_path = BASE_CVM / "2025" / "dfp_cia_aberta_DRE_con_2025.csv"
bpp_path = BASE_CVM / "2025" / "dfp_cia_aberta_BPP_con_2025.csv"
dfc_path = BASE_CVM / "2025" / "dfp_cia_aberta_DFC_MI_con_2025.csv"

print("DRE:", dre_path)
print("Balanço:", bpp_path)
print("DFC:", dfc_path)

DRE: ..\data\raw\cvm\dfp\2025\dfp_cia_aberta_DRE_con_2025.csv
Balanço: ..\data\raw\cvm\dfp\2025\dfp_cia_aberta_BPP_con_2025.csv
DFC: ..\data\raw\cvm\dfp\2025\dfp_cia_aberta_DFC_MI_con_2025.csv


In [2]:
dre = pd.read_csv(dre_path, sep=";", encoding="latin1")
bpp = pd.read_csv(bpp_path, sep=";", encoding="latin1")
dfc = pd.read_csv(dfc_path, sep=";", encoding="latin1")

print("DRE:", dre.shape)
print("Balanço:", bpp.shape)
print("DFC:", dfc.shape)

DRE: (30786, 15)
Balanço: (101380, 14)
DFC: (48500, 15)


In [3]:
EMPRESA = "VLI MULTIMODAL S.A"

dre_vli = dre[dre["DENOM_CIA"] == EMPRESA].copy()
bpp_vli = bpp[bpp["DENOM_CIA"] == EMPRESA].copy()
dfc_vli = dfc[dfc["DENOM_CIA"] == EMPRESA].copy()

print("DRE VLI:", dre_vli.shape)
print("Balanço VLI:", bpp_vli.shape)
print("DFC VLI:", dfc_vli.shape)

DRE VLI: (228, 15)
Balanço VLI: (768, 14)
DFC VLI: (354, 15)


In [4]:
dre_vli = dre_vli.drop_duplicates()
bpp_vli = bpp_vli.drop_duplicates()
dfc_vli = dfc_vli.drop_duplicates()

print("DRE VLI:", dre_vli.shape)
print("Balanço VLI:", bpp_vli.shape)
print("DFC VLI:", dfc_vli.shape)

DRE VLI: (76, 15)
Balanço VLI: (256, 14)
DFC VLI: (118, 15)


In [6]:
def extrair_contas(df, contas):
    resultado = df[df["CD_CONTA"].isin(contas)].copy()
    
    resultado = resultado[
        ["DT_FIM_EXERC", "CD_CONTA", "DS_CONTA", "VL_CONTA"]
    ]
    
    return resultado

In [7]:
contas_dre = [
    "3.01",
    "3.03",
    "3.05",
    "3.06",
    "3.07",
    "3.11"
]

dre_base = extrair_contas(dre_vli, contas_dre)

print(dre_base)

      DT_FIM_EXERC CD_CONTA  \
20334   2024-12-31     3.01   
20337   2025-12-31     3.01   
20346   2024-12-31     3.03   
20349   2025-12-31     3.03   
20418   2024-12-31     3.05   
20421   2025-12-31     3.05   
20424   2024-12-31     3.06   
20427   2025-12-31     3.06   
20466   2024-12-31     3.07   
20469   2025-12-31     3.07   
20514   2024-12-31     3.11   
20517   2025-12-31     3.11   

                                                DS_CONTA   VL_CONTA  
20334             Receita de Venda de Bens e/ou Serviços  9828861.0  
20337             Receita de Venda de Bens e/ou Serviços  9953533.0  
20346                                    Resultado Bruto  3496518.0  
20349                                    Resultado Bruto  3328413.0  
20418  Resultado Antes do Resultado Financeiro e dos ...  2794467.0  
20421  Resultado Antes do Resultado Financeiro e dos ...  2515311.0  
20424                               Resultado Financeiro -1186036.0  
20427                               

In [8]:
dre_base["ano"] = pd.to_datetime(
    dre_base["DT_FIM_EXERC"]
).dt.year

In [9]:
mapa_dre = {
    "3.01": "receita",
    "3.03": "resultado_bruto",
    "3.05": "ebit",
    "3.06": "resultado_financeiro",
    "3.07": "lucro_antes_tributos",
    "3.11": "lucro_liquido"
}

dre_base["indicador"] = dre_base["CD_CONTA"].map(mapa_dre)

In [10]:
dre_final = (
    dre_base
    .pivot(index="ano", columns="indicador", values="VL_CONTA")
    .reset_index()
)

In [11]:
contas_bpp = [
    "2.01.04",
    "2.02.01",
    "1.01.01"
]

bpp_base = extrair_contas(bpp_vli, contas_bpp)

bpp_base["ano"] = pd.to_datetime(
    bpp_base["DT_FIM_EXERC"]
).dt.year

mapa_bpp = {
    "2.01.04": "divida_curto_prazo",
    "2.02.01": "divida_longo_prazo",
    "1.01.01": "caixa"
}

bpp_base["indicador"] = bpp_base["CD_CONTA"].map(mapa_bpp)

In [12]:
bpp_final = (
    bpp_base
    .pivot(index="ano", columns="indicador", values="VL_CONTA")
    .reset_index()
)

print(bpp_final)

indicador   ano  divida_curto_prazo  divida_longo_prazo
0          2024           1653996.0           8405073.0
1          2025            473341.0           9638367.0


In [13]:
bpp_vli[
    bpp_vli["DS_CONTA"].str.contains(
        "Patrimônio Líquido",
        case=False,
        na=False
    )
][[
    "CD_CONTA",
    "DS_CONTA",
    "DT_FIM_EXERC",
    "VL_CONTA"
]].drop_duplicates()

,CD_CONTA,DS_CONTA,DT_FIM_EXERC,VL_CONTA
68240,2.03,Patrimônio Líquido Consolidado,2024-12-31,7387786.0
68243,2.03,Patrimônio Líquido Consolidado,2025-12-31,8327510.0


In [14]:
contas_bpp = [
    "1.01.01",      # caixa
    "2.01.04",      # divida de curto prazo
    "2.02.01",      # divida de longo prazo
    "2.03"          # patrimonio liquido
]

bpp_base = extrair_contas(bpp_vli, contas_bpp)

bpp_base["ano"] = pd.to_datetime(
    bpp_base["DT_FIM_EXERC"]
).dt.year

mapa_bpp = {
    "1.01.01": "caixa",
    "2.01.04": "divida_curto_prazo",
    "2.02.01": "divida_longo_prazo",
    "2.03": "patrimonio_liquido"
}

bpp_base["indicador"] = bpp_base["CD_CONTA"].map(mapa_bpp)

In [15]:
bpp_final = (
    bpp_base
    .pivot(index="ano", columns="indicador", values="VL_CONTA")
    .reset_index()
)

print(bpp_final)

indicador   ano  divida_curto_prazo  divida_longo_prazo  patrimonio_liquido
0          2024           1653996.0           8405073.0           7387786.0
1          2025            473341.0           9638367.0           8327510.0


In [19]:
bpp_final["divida_total"] = (
    bpp_final["divida_curto_prazo"]
    + bpp_final["divida_longo_prazo"]
)

bpp_final["divida_liquida"] = (
    bpp_final["divida_total"]
    - bpp_final["caixa"]
)

print(bpp_final)

indicador   ano  divida_curto_prazo  divida_longo_prazo  patrimonio_liquido  \
0          2024           1653996.0           8405073.0           7387786.0   
1          2025            473341.0           9638367.0           8327510.0   

indicador  divida_total      caixa  divida_liquida  
0            10059069.0  3210023.0       6849046.0  
1            10111708.0  2479565.0       7632143.0  


In [18]:
bpp_final["caixa"] = [3210023.0, 2479565.0]

bpp_final["divida_liquida"] = (
    bpp_final["divida_total"]
    - bpp_final["caixa"]
)

print(bpp_final)

indicador   ano  divida_curto_prazo  divida_longo_prazo  patrimonio_liquido  \
0          2024           1653996.0           8405073.0           7387786.0   
1          2025            473341.0           9638367.0           8327510.0   

indicador  divida_total      caixa  divida_liquida  
0            10059069.0  3210023.0       6849046.0  
1            10111708.0  2479565.0       7632143.0  


In [20]:
contas_dfc = [
    "6.01",
    "6.02.04",
    "6.01.01.02"
]

dfc_base = extrair_contas(dfc_vli, contas_dfc)

dfc_base["ano"] = pd.to_datetime(
    dfc_base["DT_FIM_EXERC"]
).dt.year

mapa_dfc = {
    "6.01": "cfo",
    "6.02.04": "capex",
    "6.01.01.02": "depreciacao_amortizacao"
}

dfc_base["indicador"] = dfc_base["CD_CONTA"].map(mapa_dfc)

print(dfc_base)

      DT_FIM_EXERC    CD_CONTA                               DS_CONTA  \
33118   2024-12-31        6.01  Caixa Líquido Atividades Operacionais   
33121   2025-12-31        6.01  Caixa Líquido Atividades Operacionais   
33136   2024-12-31  6.01.01.02              Depreciação e amortização   
33139   2025-12-31  6.01.01.02              Depreciação e amortização   
33394   2024-12-31     6.02.04  Aquisição de imobilizado e intangível   
33397   2025-12-31     6.02.04  Aquisição de imobilizado e intangível   

        VL_CONTA   ano                indicador  
33118  5131426.0  2024                      cfo  
33121  4640860.0  2025                      cfo  
33136  2175175.0  2024  depreciacao_amortizacao  
33139  2124895.0  2025  depreciacao_amortizacao  
33394 -3448078.0  2024                    capex  
33397 -3428442.0  2025                    capex  


In [21]:
dfc_final = (
    dfc_base
    .pivot(index="ano", columns="indicador", values="VL_CONTA")
    .reset_index()
)

print(dfc_final)

indicador   ano      capex        cfo  depreciacao_amortizacao
0          2024 -3448078.0  5131426.0                2175175.0
1          2025 -3428442.0  4640860.0                2124895.0


In [24]:
dre_vli_final = pd.DataFrame({
    "ano": [2024, 2025],
    "receita": [9828861.0, 9953533.0],
    "resultado_bruto": [3496518.0, 3328413.0],
    "ebit": [2794467.0, 2515311.0],
    "resultado_financeiro": [-1186036.0, -800313.0],
    "lucro_antes_tributos": [1608431.0, 1714998.0],
    "lucro_liquido": [2672990.0, 2774028.0]
})

print(dre_vli_final)

    ano    receita  resultado_bruto       ebit  resultado_financeiro  \
0  2024  9828861.0        3496518.0  2794467.0            -1186036.0   
1  2025  9953533.0        3328413.0  2515311.0             -800313.0   

   lucro_antes_tributos  lucro_liquido  
0             1608431.0      2672990.0  
1             1714998.0      2774028.0  


In [25]:
fundamentals_vli = (
    dre_vli_final
    .merge(bpp_final, on="ano", how="left")
    .merge(dfc_final, on="ano", how="left")
)

print(fundamentals_vli)
print(fundamentals_vli.shape)

    ano    receita  resultado_bruto       ebit  resultado_financeiro  \
0  2024  9828861.0        3496518.0  2794467.0            -1186036.0   
1  2025  9953533.0        3328413.0  2515311.0             -800313.0   

   lucro_antes_tributos  lucro_liquido  divida_curto_prazo  \
0             1608431.0      2672990.0           1653996.0   
1             1714998.0      2774028.0            473341.0   

   divida_longo_prazo  patrimonio_liquido  divida_total      caixa  \
0           8405073.0           7387786.0    10059069.0  3210023.0   
1           9638367.0           8327510.0    10111708.0  2479565.0   

   divida_liquida      capex        cfo  depreciacao_amortizacao  
0       6849046.0 -3448078.0  5131426.0                2175175.0  
1       7632143.0 -3428442.0  4640860.0                2124895.0  
(2, 16)


In [26]:
fundamentals_vli["margem_bruta"] = (
    fundamentals_vli["resultado_bruto"]
    / fundamentals_vli["receita"]
)

fundamentals_vli["margem_ebit"] = (
    fundamentals_vli["ebit"]
    / fundamentals_vli["receita"]
)

fundamentals_vli["margem_liquida"] = (
    fundamentals_vli["lucro_liquido"]
    / fundamentals_vli["receita"]
)

fundamentals_vli["roe"] = (
    fundamentals_vli["lucro_liquido"]
    / (
        fundamentals_vli["patrimonio_liquido"]
        + fundamentals_vli["patrimonio_liquido"].shift(1)
    ) / 2
)

print(
    fundamentals_vli[
        [
            "ano",
            "margem_bruta",
            "margem_ebit",
            "margem_liquida",
            "roe"
        ]
    ]
)

    ano  margem_bruta  margem_ebit  margem_liquida       roe
0  2024      0.355740     0.284312        0.271953       NaN
1  2025      0.334395     0.252705        0.278698  0.088259


In [27]:
fundamentals_vli["pl_medio"] = (
    fundamentals_vli["patrimonio_liquido"]
    + fundamentals_vli["patrimonio_liquido"].shift(1)
) / 2

fundamentals_vli["roe"] = (
    fundamentals_vli["lucro_liquido"]
    / fundamentals_vli["pl_medio"]
)

print(
    fundamentals_vli[
        [
            "ano",
            "lucro_liquido",
            "patrimonio_liquido",
            "pl_medio",
            "roe"
        ]
    ]
)

    ano  lucro_liquido  patrimonio_liquido   pl_medio       roe
0  2024      2672990.0           7387786.0        NaN       NaN
1  2025      2774028.0           8327510.0  7857648.0  0.353035


In [28]:
fundamentals_vli["divida_liquida_pl"] = (
    fundamentals_vli["divida_liquida"]
    / fundamentals_vli["patrimonio_liquido"]
)

print(
    fundamentals_vli[
        [
            "ano",
            "divida_liquida",
            "patrimonio_liquido",
            "divida_liquida_pl"
        ]
    ]
)

    ano  divida_liquida  patrimonio_liquido  divida_liquida_pl
0  2024       6849046.0           7387786.0           0.927077
1  2025       7632143.0           8327510.0           0.916498


In [29]:
fundamentals_vli["cfo_capex"] = (
    fundamentals_vli["cfo"]
    / fundamentals_vli["capex"].abs()
)

print(
    fundamentals_vli[
        [
            "ano",
            "cfo",
            "capex",
            "cfo_capex"
        ]
    ]
)

    ano        cfo      capex  cfo_capex
0  2024  5131426.0 -3448078.0   1.488199
1  2025  4640860.0 -3428442.0   1.353635


In [31]:
fundamentals_vli.to_parquet(
    "../data/processed/fundamentals_vli.parquet",
    index=False
)

print("Arquivo salvo com sucesso!")

Arquivo salvo com sucesso!


In [32]:
from pathlib import Path

arquivo = Path("../data/processed/fundamentals_vli.parquet")

print("Existe:", arquivo.exists())
print("Caminho:", arquivo)

Existe: True
Caminho: ..\data\processed\fundamentals_vli.parquet
